# 07 · Controlled prediction-collapse experiment (diagnostic; OFF by default)

Plan and rationale: `docs/COLLAPSE_EXPERIMENT_PLAN.md`. The question is why the 251-graph smoke run collapsed while
the 6-graph Part B diagnostic kept learning.

* **Normal training engine** (`Trainer.run`). Official model, loss, AdamW, clipping, labels and features. The labels
  use the c_max of the full official training split in every run.
* **Runs.** Smoke replicas (full training split, 1 epoch) in FP16 and FP32 × seeds 42/43/44, then 4 training-subset
  arms × 3 seeds in FP32, each about one smoke epoch of optimizer steps (18 runs).
* **Validation.** 6 samples of the official validation design plus 1 held-out sample of each training design (7 designs).
  The test split is never loaded.
* **Outputs.** `runs/collapse_diag/<run>/` (history, per-step `steps.csv`, last/best checkpoints) and
  `reports/07_collapse_experiment*.json`. Existing files are never overwritten; an interrupted run resumes.
* **Estimated A100 time.** About 35–65 min, plus 5–10 min of graph building. This is an estimate, not a measurement.

Cell 1 only plans: it reads the processed data and writes the plan, with no GPU work. Set `RUN_COLLAPSE_EXPERIMENT = True`
in the last cell **only after approval**. Notebook 02 is not needed.

In [ ]:
# ── Bootstrap (identical in every notebook; safe to re-run) ──────────────────────
# Mounts YOUR Google Drive (you authorise in the popup; no credentials are stored),
# unpacks the project code from Drive and puts it on the Python path.
import os, sys, zipfile, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/VeriHGN'            # persistent project storage
CODE_ZIP   = f'{DRIVE_ROOT}/code/verihgn_code.zip'        # uploaded from the laptop (tools/make_colab_bundle.py)
CODE_DIR   = '/content/verihgn-project'

if not os.path.exists(CODE_ZIP):
    raise FileNotFoundError(f'Upload verihgn_code.zip to {CODE_ZIP} first (see README, "Colab").')
stamp = f'{CODE_DIR}/.zip_mtime'
if not os.path.exists(stamp) or open(stamp).read() != str(os.path.getmtime(CODE_ZIP)):
    subprocess.run(['rm', '-rf', CODE_DIR], check=True)
    with zipfile.ZipFile(CODE_ZIP) as z:
        z.extractall(CODE_DIR)
    open(stamp, 'w').write(str(os.path.getmtime(CODE_ZIP)))
sys.path.insert(0, f'{CODE_DIR}/src')

marker = '/content/.verihgn_deps_ok'
if not os.path.exists(marker):
    from verihgn_repro.colab import install_dependencies
    install_dependencies()
    open(marker, 'w').write('ok')

from verihgn_repro.colab import DriveLayout
L = DriveLayout().ensure()
print('code:', CODE_DIR, '| drive root:', L.root)

In [ ]:
import json, time
from pathlib import Path
from verihgn_repro.config import load_config
from verihgn_repro.data.splits import load_manifest
from verihgn_repro.diagnostics import collapse_experiment as CE
SCOPE = 'official9'
CFG_DIR = Path(CODE_DIR) / 'configs'
PHASH = json.load(open(L.reports / f'03_preprocessing_{SCOPE}.json'))['prep_hash']
PROCESSED = L.processed_for(PHASH)
cfg = load_config(CFG_DIR / 'base.yaml')                  # official hyper-parameters; primary variant
assert (cfg.experiment.split, cfg.experiment.label_norm, cfg.experiment.grid_features) == \
       ('official_seed42', 'per_dataset', 'paper_eq3')
man = load_manifest(L.splits / f'{SCOPE}__{cfg.experiment.split}.json')
EXPECTED = json.load(open(CFG_DIR / 'collapse_experiment_manifest.json'))   # pre-registered selection and budgets
plan = CE.plan_experiment(PROCESSED, man, expected=EXPECTED)                  # raises if anything differs
val04 = json.load(open(L.reports / f'04_validation_{SCOPE}.json'))
assert abs(plan['c_max'] - val04['c_max'][cfg.experiment.split]) < 1e-9, 'labels differ from the validated ones'

def new_report(name):
    p = L.reports / name
    return p if not p.exists() else p.with_name(f'{p.stem}__{time.strftime("%Y%m%d_%H%M%S")}{p.suffix}')

plan_path = new_report('07_collapse_plan.json')
plan_path.write_text(json.dumps({k: v for k, v in plan.items() if k != 'sample_dirs'}, indent=1))
print('c_max', plan['c_max'], '| single-design arm:', plan['single_design'],
      '| lower-median-sparsity design (reported only):', plan['lower_median_sparsity_design'])
print('budget', plan['budget_steps'], 'steps; common budget for comparisons', plan['common_steps'])
print('validation:')
for n in plan['val']:
    print(f"  {n}  ({CE.design_of(n)})")
for r in plan['runs']:
    ds = len({CE.design_of(n) for n in r['train']})
    print(f"{r['name']:26s} train {len(r['train']):3d} graphs / {ds} designs  epochs {r['epochs']:3d}  steps {r['steps']:4d}  val {len(r['val'])}")
print('plan written to', plan_path)

In [ ]:
RUN_COLLAPSE_EXPERIMENT = False      # set True only after the experiment is approved
if RUN_COLLAPSE_EXPERIMENT:
    from verihgn_repro.data.graph import build_graph
    from verihgn_repro.data.preprocess import load_sample
    t0 = time.time()
    graphs = {}
    for name, d in plan['sample_dirs'].items():
        a, m = load_sample(Path(d))
        graphs[name] = build_graph(a, m, grid_features=cfg.experiment.grid_features,
                                   label_norm=cfg.experiment.label_norm, c_max=plan['c_max'])
    print(f'{len(graphs)} graphs built in {time.time() - t0:.0f}s')
    report_path = new_report('07_collapse_experiment.json')
    rep = CE.run_plan(plan, cfg, graphs, L.runs / 'collapse_diag', report_path=report_path)
    print(json.dumps(rep['by_arm'], indent=1, default=str))
    print('→ send back', report_path.name, 'and', plan_path.name)
else:
    print('not run (RUN_COLLAPSE_EXPERIMENT = False)')